In [11]:
from sagemaker.processing import ScriptProcessor, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.parameters import ParameterString
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker import get_execution_role

role = get_execution_role()
pipeline_session = PipelineSession()

# Parameters
input_data = ParameterString(name="InputData", default_value="s3://your-bucket/input/")
output_data = ParameterString(name="OutputData", default_value="s3://your-bucket/output/")

# ScriptProcessor
processor = ScriptProcessor(
    image_uri="683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:0.23-1-cpu-py3",
    command=["python3"],
    instance_count=1,
    instance_type="ml.m5.large",
    role=role,
    sagemaker_session=pipeline_session
)

# Chỉ định folder chứa file script
step_args = processor.run(
    code="logic",  # folder chứa your_script.py
    arguments=[
        "--input-data", input_data,
        "--output-data", output_data
    ],
    outputs=[
        ProcessingOutput(
            output_name="output_data",
            source="/opt/ml/processing/output",
            destination=output_data
        )
    ]
)

step_process = ProcessingStep(
    name="MyProcessingStep",
    step_args=step_args
)

pipeline = Pipeline(
    name="MyWorkingPipeline",
    parameters=[input_data, output_data],
    steps=[step_process],
    sagemaker_session=pipeline_session
)

pipeline.upsert(role_arn=role)


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:53                                                                                   │
│                                                                                                  │
│   50 │   sagemaker_session=pipeline_session                                                      │
│   51 )                                                                                           │
│   52                                                                                             │
│ ❱ 53 pipeline.upsert(role_arn=role)                                                              │
│   54                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:292 in upsert             │
│                                                                                                  │
│    289 │   │   │   # after fetching the config.                                                  │
│    290 │   │   │   raise ValueError("An AWS IAM role is required to create or update a Pipeline  │
│    291 │   │   try:                                                                              │
│ ❱  292 │   │   │   response = self.create(role_arn, description, tags, parallelism_config)       │
│    293 │   │   except ClientError as ce:                                                         │
│    294 │   │   │   error_code = ce.response["Error"]["Code"]                                     │
│    295 │   │   │   error_message = ce.response["Error"]["Message"]                               │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:164 in create             │
│                                                                                                  │
│    161 │   │   tags = format_tags(tags)                                                          │
│    162 │   │   tags = _append_project_tags(tags)                                                 │
│    163 │   │   tags = self.sagemaker_session._append_sagemaker_config_tags(tags, PIPELINE_TAGS_  │
│ ❱  164 │   │   kwargs = self._create_args(role_arn, description, parallelism_config)             │
│    165 │   │   update_args(                                                                      │
│    166 │   │   │   kwargs,                                                                       │
│    167 │   │   │   Tags=tags,                                                                    │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:186 in _create_args       │
│                                                                                                  │
│    183 │   │   Returns:                                                                          │
│    184 │   │   │   A keyword argument dict for calling create_pipeline.                          │
│    185 │   │   """                                                                               │
│ ❱  186 │   │   pipeline_definition = self.definition()                                           │
│    187 │   │   kwargs = dict(                                                                    │
│    188 │   │   │   PipelineName=self.name,                                                       │
│    189 │   │   │   RoleArn=role_arn,                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/